# 2D Drone Pathfinding RL — Parameter Sensitivity Analysis
**Phase 8 | Dr. Segal §9 | `drone-rl v1.00`**

This notebook quantifies how the three core Q-learning hyperparameters affect
convergence speed and final policy quality on a **5×5 grid** (start (0,0) →
goal (4,4), one BUILDING obstacle at (2,2)).

| Hyperparameter | Values tested |
|---|---|
| α (learning rate) | 0.01, 0.05, 0.1, 0.5, 1.0 |
| γ (discount factor) | 0.0, 0.5, 0.9, 0.95, 0.99 |
| ε (initial exploration) | 0.0, 0.05, 0.1, 0.2, 0.5 |

Each (α, γ, ε) combination is run **5 independent times** (different seeds).
Results are saved to `results/sensitivity_analysis.json`.

## Core Equations

### Bellman Q-Learning Update

$$Q(s,a) \leftarrow Q(s,a) + \alpha\bigl[R(s,a) + \gamma \max_{a'} Q(s',a') - Q(s,a)\bigr]$$

where:
- $Q(s,a)$ = current Q-value for state $s$, action $a$
- $\alpha \in (0, 1]$ = learning rate (how fast to update estimates)
- $R(s,a)$ = immediate reward received after taking action $a$ in state $s$
- $\gamma \in [0, 1)$ = discount factor (importance of future rewards)
- $\max_{a'} Q(s',a')$ = best Q-value achievable from next state $s'$

### ε-Greedy Policy

$$\pi(a \mid s) = \begin{cases}
1 - \varepsilon + \frac{\varepsilon}{|A|} & \text{if } a = \arg\max_a Q(s,a) \\
\frac{\varepsilon}{|A|} & \text{otherwise}
\end{cases}$$

### Reward Function

$$R(s,a) = \begin{cases}
+100 & \text{if } s' = \text{goal} \\
-1   & \text{empty cell step} \\
-10  & \text{building collision} \\
-100 & \text{trap hit} \\
-10  & \text{crosswind penalty}
\end{cases}$$

In [ ]:
import itertools
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Make sure drone_rl is importable when running from notebooks/
sys.path.insert(0, str(Path(__file__ if "__file__" in dir() else ".").parent.parent / "src"))
from drone_rl.sdk import DroneRLSDK
from drone_rl.types.grid import CellType, Coordinate, GridState
from drone_rl.types.rl import Hyperparameters

RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

# Reproducibility
RNG_BASE_SEED = 42
N_SEEDS = 5
N_EPISODES = 200  # fast run; increase to 1000 for publication-quality results
GRID = GridState(
    rows=5,
    cols=5,
    cells={(2, 2): CellType.BUILDING},
    start_pos=Coordinate(0, 0),
    goal_pos=Coordinate(4, 4),
)
print("Setup complete. Grid: 5×5, obstacle at (2,2), episodes:", N_EPISODES)

In [ ]:
ALPHAS = [0.01, 0.05, 0.1, 0.5, 1.0]
GAMMAS = [0.0, 0.5, 0.9, 0.95, 0.99]
EPSILONS = [0.0, 0.05, 0.1, 0.2, 0.5]

combos = list(itertools.product(ALPHAS, GAMMAS, EPSILONS))
print(f"Total combinations: {len(combos)}  ×  {N_SEEDS} seeds = {len(combos) * N_SEEDS} runs")

In [ ]:
def run_experiment(alpha, gamma, epsilon, seed):
    hp = Hyperparameters(
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon,
        epsilon_decay=0.995,
        epsilon_min=0.01,
        max_steps_per_episode=50,
        total_episodes=N_EPISODES,
        random_seed=seed,
    )
    sdk = DroneRLSDK(hp=hp)
    sdk.create_environment(GRID)
    sdk.train(num_episodes=N_EPISODES)
    stats = sdk.get_episode_stats()
    last50 = stats[-50:] if len(stats) >= 50 else stats
    success_rate = sum(1 for r in last50 if r["terminal_reason"] == "GOAL") / len(last50)
    # convergence_ep: first episode where 10-run rolling success ≥ 0.8, else N_EPISODES
    conv_ep = N_EPISODES
    for i in range(10, len(stats)):
        window = stats[i - 10 : i]
        if sum(1 for r in window if r["terminal_reason"] == "GOAL") / 10 >= 0.8:
            conv_ep = i
            break
    mean_reward = np.mean([r["total_reward"] for r in last50])
    return {"success_rate": success_rate, "convergence_ep": conv_ep, "mean_reward": mean_reward}


rows = []
for i, (a, g, e) in enumerate(combos):
    seed_results = [run_experiment(a, g, e, RNG_BASE_SEED + s) for s in range(N_SEEDS)]
    rows.append(
        {
            "alpha": a,
            "gamma": g,
            "epsilon": e,
            "success_rate_mean": np.mean([r["success_rate"] for r in seed_results]),
            "success_rate_std": np.std([r["success_rate"] for r in seed_results]),
            "conv_ep_mean": np.mean([r["convergence_ep"] for r in seed_results]),
            "mean_reward_mean": np.mean([r["mean_reward"] for r in seed_results]),
        }
    )
    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(combos)} combinations done…")

df = pd.DataFrame(rows)
print(f"Done. DataFrame shape: {df.shape}")
df.head()

In [ ]:
out = {
    "metadata": {
        "n_episodes": N_EPISODES,
        "n_seeds": N_SEEDS,
        "grid": "5x5-obstacle-22",
    },
    "results": df.to_dict(orient="records"),
}
out_path = RESULTS_DIR / "sensitivity_analysis.json"
out_path.write_text(json.dumps(out, indent=2))
print(f"Saved {len(df)} rows → {out_path}")

In [ ]:
pivot = df.groupby(["alpha", "gamma"])["success_rate_mean"].mean().unstack()

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(g) for g in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(a) for a in pivot.index])
ax.set_xlabel("γ (discount factor)", fontsize=12)
ax.set_ylabel("α (learning rate)", fontsize=12)
ax.set_title("Mean Success Rate (last 50 eps) vs (α, γ)\n(averaged over all ε)", fontsize=13)
plt.colorbar(im, ax=ax, label="Success Rate")
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "heatmap_alpha_gamma.png"), dpi=120)
plt.show()
print("Heatmap saved.")

In [ ]:
groups = [df[df.epsilon == e]["success_rate_mean"].values for e in EPSILONS]

fig, ax = plt.subplots(figsize=(7, 4))
bp = ax.boxplot(
    groups,
    patch_artist=True,
    boxprops={"facecolor": "#90caf9"},
    medianprops={"color": "navy", "linewidth": 2},
)
ax.set_xticklabels([f"ε={e}" for e in EPSILONS])
ax.set_ylabel("Mean Success Rate (last 50 eps)")
ax.set_title("Success Rate Distribution by Initial ε")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "boxplot_epsilon.png"), dpi=120)
plt.show()

In [ ]:
def get_rolling_success(alpha, gamma=0.9, epsilon=0.1, seed=42, window=10):
    hp = Hyperparameters(
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon,
        epsilon_decay=0.995,
        epsilon_min=0.01,
        max_steps_per_episode=50,
        total_episodes=N_EPISODES,
        random_seed=seed,
    )
    sdk = DroneRLSDK(hp=hp)
    sdk.create_environment(GRID)
    sdk.train(num_episodes=N_EPISODES)
    stats = sdk.get_episode_stats()
    success = [1 if r["terminal_reason"] == "GOAL" else 0 for r in stats]
    return pd.Series(success).rolling(window).mean().values


fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#e53935", "#fb8c00", "#43a047", "#1e88e5", "#8e24aa"]
for alpha, color in zip(ALPHAS, colors, strict=False):
    curve = get_rolling_success(alpha)
    ax.plot(curve, label=f"α={alpha}", color=color, linewidth=1.8)
ax.set_xlabel("Episode")
ax.set_ylabel("10-Episode Rolling Success Rate")
ax.set_title("Convergence Trajectories by α (γ=0.9, ε=0.1)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "convergence_by_alpha.png"), dpi=120)
plt.show()

In [ ]:
summary = (
    df.groupby("alpha")
    .agg(
        success_mean=("success_rate_mean", "mean"),
        success_std=("success_rate_mean", "std"),
        conv_ep_mean=("conv_ep_mean", "mean"),
    )
    .round(4)
)
print("\n=== Summary by α ===")
print(summary.to_string())

summary_g = (
    df.groupby("gamma")
    .agg(
        success_mean=("success_rate_mean", "mean"),
        conv_ep_mean=("conv_ep_mean", "mean"),
    )
    .round(4)
)
print("\n=== Summary by γ ===")
print(summary_g.to_string())

## Key Findings

### Learning Rate (α)
- **High α (0.5–1.0)** converges faster initially but can oscillate around the optimal policy.
- **Low α (0.01)** learns slowly but produces smoother, more stable convergence.
- **Recommended:** α ≈ 0.1 balances speed and stability for the 5×5 grid.

### Discount Factor (γ)
- **γ = 0.0** yields myopic behaviour — the agent ignores future rewards entirely.
- **γ ≥ 0.9** is required for reliable navigation to a distant goal.
- **Recommended:** γ = 0.95 for this task (goal ≈ 8 steps from start).

### Initial Exploration (ε)
- **ε = 0.0** exploits from the start — converges only if the initial Q-table happens to guide the agent to the goal.
- **ε ≥ 0.1** ensures sufficient exploration on a sparse reward grid.
- **Recommended:** ε = 0.5 with ε-decay to ε_min = 0.01.

### Parameter Interactions
- High α + high γ (e.g. 1.0 + 0.99) can cause instability due to large target oscillations.
- Low ε + low γ results in poor exploration and myopic policies simultaneously.

### Recommended Configuration

| Parameter | Value |
|---|---|
| α | 0.1 |
| γ | 0.95 |
| ε₀ | 0.5 |
| ε-decay | 0.995 |
| ε_min | 0.01 |